# LAB 3c: BigQuery ML ディープ ニューラル ネットワーク モデル

**学習目標**

1. BigQuery ML を使用して DNN モデルを作成および評価する
1. ML.TRANSFORM による特徴量エンジニアリングを使用して DNN モデルを作成および評価する
1. BigQuery の ML.PREDICT を使用して予測を計算する


## はじめに 
このノートブックでは、まず特徴量エンジニアリングを行わずに、次に前のラボの特徴量エンジニアリングを使用して、BigQuery ML を用いて生まれる前の赤ちゃんの体重を予測する複数のディープ ニューラル ネットワーク（DNN）モデルを作成します。

BigQuery ML を使用して、BigQuery の ML.TRANSFORM を用いた特徴量エンジニアリングあり/なしで DNN モデルを作成および評価し、BigQuery の ML.PREDICT を使用して予測を計算します。復習が必要な場合は、ノートブック [BQML ベースライン モデル](../solutions/3a_bqml_baseline_babyweight.ipynb) でベースライン モデルを作成した方法や、ノートブック [特徴量エンジニアリングを含む BQML 線形モデル](../solutions/3b_bqml_linear_transform_babyweight.ipynb) で線形モデルと特徴量エンジニアリングを組み合わせた方法を振り返ることができます。

## テーブルが存在することの確認

次のセルを実行して、データセットとデータテーブルが以前に作成されていることを確認します。作成されていない場合は、ラボ [1b_prepare_data_babyweight](../solutions/1b_prepare_data_babyweight.ipynb) に戻って作成してください。

In [ ]:
%load_ext google.cloud.bigquery

In [2]:
%%bigquery
-- LIMIT 0 is a free query; this allows us to check that the table exists.
SELECT * FROM babyweight.babyweight_data_train
LIMIT 0

,weight_pounds,is_male,mother_age,plurality,gestation_weeks


In [3]:
%%bigquery
-- LIMIT 0 is a free query; this allows us to check that the table exists.
SELECT * FROM babyweight.babyweight_data_eval
LIMIT 0

,weight_pounds,is_male,mother_age,plurality,gestation_weeks


## モデル 4: DNN_REGRESSOR を使用したモデルの複雑さの向上

DNN_REGRESSOR は、前のラボで使用してきた LINEAR_REG と比較して新しい回帰 model_type です。

* MODEL_TYPE="DNN_REGRESSOR"

* hidden_units: レイヤごとの隠れユニットのリスト。すべてのレイヤは全結合（fully connected）です。配列内の要素数が隠れレイヤの数になります。hidden_units のデフォルト値は [Min(128, N / (𝜶(Ni+No)))]（1 つの隠れレイヤ）です。ここで、N はトレーニング データサイズ、Ni と No はそれぞれ入力レイヤと出力レイヤのユニット数、𝜶 は値 10 の定数です。このルールのブレークポイントにより、モデルが過学習（オーバーフィッティング）しないようになります。なお、現在モデルサイズは 256MB に制限されています。

* dropout: トレーニング中に特定の座標をドロップする確率。ドロップアウトは、DNN における過学習を回避するための非常に一般的な手法です。デフォルト値はゼロであり、トレーニング中に座標がドロップされないことを意味します。

* batch_size: 各サブイテレーションでネットワークのトレーニングに提供されるサンプル数。トレーニング速度と収束のバランスをとるため、デフォルト値は Min(1024, num_examples) です。各サブイテレーションですべてのトレーニング データを提供すると収束の問題が発生する可能性があり、推奨されません。

### DNN_REGRESSOR モデルの作成

BigQuery で `MODEL_TYPE=DNN_REGRESSOR` を使用し、それぞれ 64 個と 32 個のニューロンを持つ 2 つの隠れレイヤ（`HIDDEN_UNITS=[64, 32]`）とバッチサイズ 32（`BATCH_SIZE=32`）を指定して、DNN 回帰モデルをトレーニングしましょう:

In [4]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_4
OPTIONS (
    MODEL_TYPE="DNN_REGRESSOR",
    HIDDEN_UNITS=[64, 32],
    BATCH_SIZE=32,
    INPUT_LABEL_COLS=["weight_pounds"],
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks
FROM
    babyweight.babyweight_data_train

""


### トレーニング情報の取得と評価

まずトレーニング統計情報を確認しましょう。

In [5]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.model_4)

,training_run,iteration,loss,eval_loss,learning_rate,duration_ms
0,0,4,1.738382,2.713294,0.1,644721
1,0,3,2.405464,2.604605,0.1,606809
2,0,2,0.732892,2.641865,0.1,599237
3,0,1,1.014293,2.191636,0.1,618610
4,0,0,1.001003,2.943201,0.1,830424


次に、評価データセットでトレーニング済みモデルを評価しましょう。

In [6]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_4,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks
    FROM
        babyweight.babyweight_data_eval
    ))

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,0.803307,1.065722,0.018036,0.658235,0.387528,0.390166


評価の `mean_squared_error` を使用してモデルの RMSE を計算しましょう。

In [7]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.model_4,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks
    FROM
        babyweight.babyweight_data_eval
    ))

,rmse
0,1.032338


## 最終モデル: TRANSFORM 句の適用

予測を実行する前に、前のノートブックで行ったように、特徴量セット全体を TRANSFORM 句にカプセル化する必要があります。これにより、クエリを変更することなく、トレーニングと予測に同じ変換を適用できます。

最終モデルに TRANSFORM 句を適用し、クエリを実行しましょう。

In [1]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.final_model

TRANSFORM(
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    ML.FEATURE_CROSS(
        STRUCT(
            is_male,
            ML.BUCKETIZE(
                mother_age,
                GENERATE_ARRAY(15, 45, 1)
            ) AS bucketed_mothers_age,
            plurality,
            ML.BUCKETIZE(
                gestation_weeks,
                GENERATE_ARRAY(17, 47, 1)
            ) AS bucketed_gestation_weeks)
    ) AS crossed)

OPTIONS (
    MODEL_TYPE="DNN_REGRESSOR",
    HIDDEN_UNITS=[64, 32],
    BATCH_SIZE=32,
    INPUT_LABEL_COLS=["weight_pounds"],
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    *
FROM
    babyweight.babyweight_data_train

""


まずトレーニング統計情報を確認しましょう。

In [2]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.final_model)

,training_run,iteration,loss,eval_loss,learning_rate,duration_ms
0,0,3,0.681415,2.200278,0.1,147373
1,0,2,0.970840,1.803952,0.1,148089
2,0,1,0.704043,1.594089,0.1,146653
3,0,0,1.400467,2.130462,0.1,216121


次に、評価データセットでトレーニング済みモデルを評価しましょう。

In [3]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.final_model,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.230722,2.346158,0.037347,1.055479,-0.34834,0.215212


評価の `mean_squared_error` を使用してモデルの RMSE を計算しましょう。

In [4]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.final_model,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

,rmse
0,1.531717


## 最終モデルによる予測


モデルを評価したので、次のステップは BigQuery の `ML.PREDICT` 関数を使用して、生まれる前の赤ちゃんの体重を予測することです。

### 元のデータセットのサンプルを使用した最終モデルからの予測

In [5]:
%%bigquery
SELECT
    *
FROM
    ML.PREDICT(MODEL babyweight.final_model,
    (
    SELECT
        "true" AS is_male,
        32 AS mother_age,
        "Twins(2)" AS plurality,
        30 AS gestation_weeks
    ))

,predicted_weight_pounds,is_male,mother_age,plurality,gestation_weeks
0,3.675936,true,32,Twins(2),30


### シミュレートされたデータセットのサンプルを使用した上記の予測クエリの変更

上記で作成した特徴量の値を使用しますが、is_male を "Unknown" に、plurality を "Multiple(2+)" に設定してください。これは、性別や正確な胎数が不明である状況をシミュレートしています。

In [6]:
%%bigquery
SELECT
    *
FROM
    ML.PREDICT(MODEL babyweight.final_model,
    (
    SELECT
        "Unknown" AS is_male,
        32 AS mother_age,
        "Multiple(2+)" AS plurality,
        30 AS gestation_weeks
    ))

,predicted_weight_pounds,is_male,mother_age,plurality,gestation_weeks
0,3.477121,Unknown,32,Multiple(2+),30


## ラボのまとめ: 
このラボでは、BigQuery ML を使用して、BigQuery の ML.TRANSFORM を用いた特徴量エンジニアリングあり/なしで DNN モデルを作成および評価し、BigQuery の ML.PREDICT を使用して予測を計算しました。

Copyright 2020 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.